# Amprente județene

**Maramureș nu sună ca Dobrogea.**

Fiecare județ are un profil distinct: ce procent din străzi poartă prefix sfânt, câte sunt numerotate anonim, cât de diversă este nomenclatura (entropie). Acest notebook permite compararea județelor pe aceste dimensiuni.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.rcParams.update({'font.family':'serif','figure.dpi':130,
                     'axes.spines.top':False,'axes.spines.right':False})
ACCENT, INK, MUTED = '#C04F35', '#15171A', '#6E6E70'

conn = sqlite3.connect('../data/streets.db')
conn.row_factory = sqlite3.Row
print('Connected.')

## 1. Profilul de bază per județ

In [ ]:
df = pd.read_sql("""
    SELECT judet,
           COUNT(*) AS total_streets,
           ROUND(100.0*SUM(is_saint)/COUNT(*),2) AS saint_pct,
           ROUND(100.0*SUM(is_numeric)/COUNT(*),2) AS numeric_pct,
           ROUND(100.0*SUM(CASE WHEN p.gender='F' THEN 1 ELSE 0 END)/COUNT(*),2) AS female_pct
    FROM streets_dedup sd
    LEFT JOIN persons p ON p.core_name_norm = sd.core_name_norm
    GROUP BY judet
    ORDER BY total_streets DESC
""", conn)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = [
    ('saint_pct',   '% sfinți',    INK),
    ('numeric_pct', '% anonim',    MUTED),
    ('female_pct',  '% feminin',   ACCENT),
]
for ax, (col, title, color) in zip(axes, metrics):
    sorted_df = df.sort_values(col, ascending=False)
    ax.bar(sorted_df['judet'], sorted_df[col], color=color, alpha=0.8)
    ax.set_title(title, fontsize=12)
    ax.set_ylabel('%')
    plt.sca(ax)
    plt.xticks(rotation=90, fontsize=7)

plt.suptitle('Profilul județelor — 3 dimensiuni', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 2. Diversitatea nomenclaturii (entropie Shannon)

In [ ]:
df_names = pd.read_sql("""
    SELECT judet, name_normalized, COUNT(*) AS cnt
    FROM streets_dedup
    WHERE is_numeric=0 AND core_name IS NOT NULL
    GROUP BY judet, name_normalized
""", conn)

def shannon_entropy(counts):
    total = counts.sum()
    probs = counts / total
    return -(probs * np.log2(probs + 1e-12)).sum()

entropy = df_names.groupby('judet')['cnt'].apply(shannon_entropy).reset_index()
entropy.columns = ['judet','entropy']
entropy = entropy.merge(df[['judet','total_streets']], on='judet')
entropy = entropy.sort_values('entropy', ascending=False)

fig, ax = plt.subplots(figsize=(13, 5))
colors = [ACCENT if e > entropy['entropy'].median() else INK for e in entropy['entropy']]
ax.bar(entropy['judet'], entropy['entropy'], color=colors, alpha=0.8)
ax.axhline(entropy['entropy'].median(), color=MUTED, linestyle='--',
           label=f'Mediană: {entropy["entropy"].median():.2f}')
ax.set_title('Diversitatea nomenclaturii (entropie Shannon) per județ', fontsize=13)
ax.set_ylabel('Entropie (biți)')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('\nTop 5 cele mai diverse județe:')
print(entropy.head())
print('\nTop 5 cele mai uniforme județe:')
print(entropy.tail())

## 3. Cel mai frecvent nume per județ

In [ ]:
df_modal = pd.read_sql("""
    SELECT judet, name_normalized, cnt FROM (
        SELECT judet, name_normalized, COUNT(*) AS cnt,
               ROW_NUMBER() OVER (PARTITION BY judet ORDER BY COUNT(*) DESC) AS rn
        FROM streets_dedup
        WHERE is_numeric=0 AND core_name IS NOT NULL
        GROUP BY judet, name_normalized
    ) WHERE rn=1
    ORDER BY cnt DESC
""", conn)

print(df_modal.to_string(index=False))

## 4. Acoperire OSM pe județ

In [ ]:
df_osm = pd.read_sql("""
    SELECT sd.judet,
           COUNT(DISTINCT sd.rowid) AS registry_streets,
           COUNT(DISTINCT m.street_id) AS matched_streets,
           ROUND(100.0*COUNT(DISTINCT m.street_id)/COUNT(DISTINCT sd.rowid),1) AS osm_pct
    FROM streets_dedup sd
    LEFT JOIN street_osm_matches m ON m.street_id = sd.id
    GROUP BY sd.judet
    ORDER BY osm_pct DESC
""", conn)

if df_osm['matched_streets'].sum() > 0:
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.bar(df_osm['judet'], df_osm['osm_pct'], color=INK, alpha=0.75)
    ax.axhline(df_osm['osm_pct'].mean(), color=ACCENT, linestyle='--',
               label=f'Medie: {df_osm["osm_pct"].mean():.1f}%')
    ax.set_title('Acoperire OSM pe județ (% din registru găsit în OpenStreetMap)', fontsize=13)
    ax.set_ylabel('%')
    ax.legend()
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('Nu există date OSM în baza de date curentă.')

In [ ]:
conn.close()